# Unified Notebook 
* This notebook will act as a gateway to generate the diagrams used in the report
* To train the TPCRPs individually, please refer to the TPCRP_Algorithms directory, and execute the .py's individually, they will train for 500 epochs, though this can be reduced from the __init__ method
* **Please note** the majority of graphs are already present in the "Evaluation Metrics" Directory 

* Thus the training pipeline will be found in TPCRP_Algorithm

In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt  
import seaborn as sns  
import random 

import torch 

import torchvision 
import torchvision.transforms as transforms 

from sklearn.datasets import make_blobs
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans


from sklearn.decomposition import PCA 


from Notebooks.supervised_training import train_supervised

In [ ]:
budgets = [10,20,40,80]
seeds = [0,1,2,3,4]

# Unsupervised
typiclust_unsup = {
    B: np.load(f"budget_results/unsupervised_budget_results/typiclust_B{B}.npy")
    for B in budgets
}

# Self-supervised (TPCRP)
typiclust_ssl = {
    B: np.load(f"budget_results/unmodified_budget_results/typiclust_B{B}.npy")
    for B in budgets
}

modified_typiclust = {
    B : np.load(f"budget_results/modified_budget_results/typiclust_B{B}.npy")
    for B in budgets
}

# Fully supervised
typiclust_sup = {
    B: np.load(f"budget_results/supervised_budget_results/typiclust_B{B}.npy")
    for B in budgets
}


# 'Features' from SSL Unmodified ( self-supervised ) - not possible to send via GitHub due to large size
# 'Features' can be independently ( and reliably ) obtained by running the TPCRP_Algorithm/Modified_TPCRP_Algorithm.py
#features = np.load("budget_results/modified_budget_results/features.npy")

# Cross-Framework-Comparison - for comparison between the different frameworks used 

In [ ]:
N = 50000 # CIFAR-10 SIZE
def generate_random_indices(seed, B):
    rng = np.random.RandomState(seed)
    return rng.choice(N, size=B, replace=False)

# Please note : The cell below will take some time to complete 

In [ ]:
results = []

for seed in seeds:
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    for B in budgets:

        # --- Unsupervised TypiClust ---
        idx = typiclust_unsup[B]
        acc, loss, runtime = train_supervised(idx, epochs=50)
        results.append({"method": "Unsupervised", "budget": B, "seed": seed,
                        "accuracy": acc, "runtime": runtime})

        # --- Self-supervised TypiClust ---
        idx = typiclust_ssl[B]
        acc, loss, runtime = train_supervised(idx, epochs=50)
        results.append({"method": "Self-supervised", "budget": B, "seed": seed,
                        "accuracy": acc, "runtime": runtime})

        # --- Fully supervised TypiClust ---
        idx = typiclust_sup[B]
        acc, loss, runtime = train_supervised(idx, epochs=50)
        results.append({"method": "Supervised", "budget": B, "seed": seed,
                        "accuracy": acc, "runtime": runtime})

        # --- Random baseline ---
        idx = generate_random_indices(seed, B)
        acc, loss, runtime = train_supervised(idx, epochs=50)
        results.append({"method": "Random", "budget": B, "seed": seed,
                        "accuracy": acc, "runtime": runtime})

df = pd.DataFrame(results)
df.to_csv("typiclust_framework_comparison.csv", index=False)
df


In [ ]:
plt.figure(figsize=(8,6))
sns.lineplot(data=df, x="budget", y="accuracy", hue="method", marker="o")
plt.title("Accuracy vs Budget Across Frameworks")
plt.grid(True)
plt.show()


In [ ]:
df_random = df[df["method"] == "Random"].set_index(["budget", "seed"])

df["acc_diff"] = df.apply(
    lambda row: row["accuracy"] - df_random.loc[(row["budget"], row["seed"]), "accuracy"],
    axis=1
)

plt.figure(figsize=(8,6))
sns.lineplot(data=df[df["method"] != "Random"],
             x="budget", y="acc_diff", hue="method", marker="o")
plt.axhline(0, color="black", linestyle="--")
plt.title("Accuracy Difference vs Random Baseline")
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(8,6))
sns.histplot(typiclust_unsup[10], label="Unsupervised", kde=True)
sns.histplot(typiclust_ssl[10], label="Self-supervised", kde=True)
sns.histplot(typiclust_sup[10], label="Supervised", kde=True)
plt.legend()
plt.title("Distribution of Selected Indices (B=10)")
plt.show()


# HDBSCAN Notebook - for the modified Algorithm

In [ ]:
pca = PCA(n_components=2) # 2D
#proj = pca.fit_transform(features) # Disabled due to features not being able to be committed ( GitHub storage restrictions )

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    ),
])

dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)


In [ ]:
def plot_typiclust_visuals(indices, proj, dataset, B, title_prefix):
    limit = 20

    # -----------------------------
    # 1. PCA manifold plot
    # -----------------------------
    plt.figure(figsize=(7,7))
    plt.scatter(proj[:,0], proj[:,1], s=5, alpha=0.3, label="All points")
    plt.scatter(proj[indices,0], proj[indices,1], s=40, color="red", label="Selected")
    plt.title(f"{title_prefix} — PCA Manifold (B={B})")
    plt.legend()
    plt.show()

    # -----------------------------
    # 2. Index scatter
    # -----------------------------
    plt.figure(figsize=(10,2))
    plt.scatter(indices, [0]*len(indices), s=30)
    plt.title(f"{title_prefix} — Selected Indices (B={B})")
    plt.yticks([])
    plt.xlabel("Dataset Index")
    plt.show()

    # -----------------------------
    # 3. CIFAR‑10 thumbnails
    # -----------------------------
    plt.figure(figsize=(12,6))
    for i, idx in enumerate(indices[:limit]):
        img, label = dataset[idx]
        img = img.permute(1,2,0).numpy()
        img = (img * 0.2470 + 0.4914).clip(0,1)

        plt.subplot(4,5,i+1)
        plt.imshow(img)
        plt.title(f"{label}")
        plt.axis("off")

    plt.suptitle(f"{title_prefix} — First {limit} Selected Samples (B={B})")
    plt.show()


In [ ]:
for B in budgets:
    print(f"\n=/\ BUDGET {B} /\=")

    plot_typiclust_visuals(typiclust_unsup[B], proj, dataset, B, "Unsupervised TypiClust")
    plot_typiclust_visuals(typiclust_ssl[B], proj, dataset, B, "Self-Supervised TypiClust")
    plot_typiclust_visuals(typiclust_sup[B], proj, dataset, B, "Supervised TypiClust")
    #plot_typiclust_visuals(modified_typiclust[B], proj, dataset, B, "Modified TypiClust")

    # PLEASE NOTE : the 'features' accessed to be used cannot be saved on the GitHub Repository due to a size > 100MB ( Added to .gitignore )
    # The Modified TypiClust consequently cannot be accessed - so I have commented it out 




# Uncertainity-Baseline Plot Design 